# Análise de Vendedores com Alta Quantidade de Vendas e Baixa Avaliação

Este notebook investiga vendedores da base Olist com volume relevante de pedidos e desempenho baixo de satisfação.
O foco é entender o peso desse grupo no negócio por meio de métricas de pedidos, receita e participação de vendedores.

**Escopo desta versão:** preparação da base, segmentação de detratores e resumo quantitativo (sem visualizações).

**Fonte dos dados:** [Olist E-Commerce Public Dataset - Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

**Fluxo da análise**
1. Importação das bibliotecas e carregamento das tabelas principais.
2. Limpeza mínima por domínio (pedidos, reviews, itens e vendedores).
3. Construção da base analítica por vendedor.
4. Segmentação em detratores vs. não detratores.
5. Consolidação do impacto percentual em pedidos, receita e base de sellers.

---
## 1. Importação e carregamento das tabelas

Nesta etapa, importamos as bibliotecas e carregamos as tabelas essenciais para a análise de desempenho dos vendedores.
Os arquivos CSV estão no diretório `data/`.

- **order_items**: relacionamento entre pedidos, produtos e vendedores.
- **order_reviews**: notas de satisfação dos clientes por pedido.
- **orders**: status e informações de datas dos pedidos.
- **sellers**: dados cadastrais de localização dos vendedores.
- **products**: metadados de produtos usados em extensões futuras da análise.

In [64]:
import pandas as pd
from pathlib import Path

# DATA_DIR aponta para a pasta onde estão os CSVs do projeto
# Se os arquivos estiverem na pasta correta, basta clicar em "Run" nesta célula
DATA_DIR = Path("data")

# Carregamento das tabelas principais da análise
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")

# Conferência rápida: quantas linhas foram lidas em cada tabela
# Isso ajuda a confirmar que o carregamento funcionou
print('Linhas carregadas:')
print(f'  orders:      {len(orders):>7,}')
print(f'  order_items: {len(order_items):>7,}')
print(f'  reviews:     {len(reviews):>7,}')
print(f'  products:    {len(products):>7,}')
print(f'  sellers:     {len(sellers):>7,}')

Linhas carregadas:
  orders:       99,441
  order_items: 112,650
  reviews:      99,224
  products:     32,951
  sellers:       3,095


---
## 2. Limpeza das tabelas

A limpeza é aplicada separadamente por tabela para facilitar rastreabilidade.
Cada regra abaixo busca reduzir ruído sem perder representatividade estatística.

- **orders**: mantém apenas pedidos entregues com data de entrega válida.
- **reviews**: mantém notas válidas e remove duplicidade de `order_id`.
- **order_items**: remove itens sem `seller_id` e com preço inválido.
- **sellers**: mantém apenas registros com cidade e estado preenchidos.

In [57]:
# -------------------- Limpeza da tabela de pedidos (orders) --------------------
# Objetivo: manter apenas pedidos realmente entregues, pois só eles fazem sentido
# para análise de satisfação do cliente.

# 1) Filtra apenas pedidos com status "delivered" (entregue)
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

# 2) Remove pedidos sem data de entrega ao cliente
# Sem essa data, não conseguimos confirmar a entrega de forma confiável
orders_clean = orders_clean.dropna(subset=['order_delivered_customer_date'])

# 3) Mostra o efeito da limpeza
print(f'orders original:  {len(orders):,} linhas')
print(f'orders limpo:     {len(orders_clean):,} linhas')

orders original:  99,441 linhas
orders limpo:     96,470 linhas


In [65]:
# ------------------- Limpeza da tabela de avaliações (reviews) -------------------
# Objetivo: garantir que a nota usada na média do vendedor seja válida e sem repetição.

# 1) Remove linhas sem nota de avaliação
reviews_clean = reviews.dropna(subset=['review_score']).copy()

# 2) Mantém apenas notas dentro da escala oficial (1 a 5)
reviews_clean = reviews_clean[reviews_clean['review_score'].between(1, 5)]

# 3) Remove duplicidades de order_id
# Se houver mais de uma avaliação para o mesmo pedido, mantemos a primeira
reviews_clean = reviews_clean.drop_duplicates(subset='order_id', keep='first')

# 4) Mostra o efeito da limpeza
print(f'reviews original: {len(reviews):,} linhas')
print(f'reviews limpo:    {len(reviews_clean):,} linhas')

reviews original: 99,224 linhas
reviews limpo:    98,673 linhas


In [59]:
# ------------------- Limpeza da tabela de itens do pedido -------------------
# Objetivo: manter itens que permitem identificar o vendedor e calcular receita.

# 1) Remove itens sem seller_id (sem vendedor identificado)
items_clean = order_items.dropna(subset=['seller_id']).copy()

# 2) Remove itens com preço zero ou negativo (inconsistentes para análise de receita)
items_clean = items_clean[items_clean['price'] > 0]

# 3) Mostra o efeito da limpeza
print(f'order_items original: {len(order_items):,} linhas')
print(f'order_items limpo:    {len(items_clean):,} linhas')

order_items original: 112,650 linhas
order_items limpo:    112,650 linhas


In [66]:
# ------------------- Limpeza da tabela de vendedores (sellers) -------------------
# Objetivo: manter dados mínimos de localização para permitir análises por região e garantir 
# que todos os vendedores tenham informações relevantes para análises posteriores.

# Remove vendedores sem cidade ou estado preenchidos
sellers_clean = sellers.dropna(subset=['seller_city', 'seller_state']).copy()

# Mostra o efeito da limpeza
print(f'sellers original: {len(sellers):,} linhas')
print(f'sellers limpo:    {len(sellers_clean):,} linhas')

sellers original: 3,095 linhas
sellers limpo:    3,095 linhas


---
## 3. Montagem da base analítica por vendedor

As bases limpas são cruzadas para formar uma visão consolidada por `seller_id`.

**Métricas calculadas por vendedor:**
- `total_pedidos`: quantidade de pedidos únicos entregues.
- `media_nota`: média do `review_score`.
- `receita_total`: soma de `price` dos itens vendidos.

Resultado desta etapa: dataframe `seller_stats`, base principal para a segmentação de risco.

In [67]:
# ------------------- Montagem da base analítica por vendedor -------------------
# Nesta célula, juntamos tabelas diferentes para chegar a uma visão única por seller_id.

# 1) Mantém apenas itens que pertencem a pedidos entregues
# (join interno: só fica o que existe em ambas as tabelas)
items_delivered = items_clean.merge(
    orders_clean[['order_id']],
    on='order_id',
    how='inner'
)

# 2) Traz a nota (review_score) para cada pedido entregue
# Usamos left join para não perder itens; pedidos sem nota ficam com NaN
items_with_review = items_delivered.merge(
    reviews_clean[['order_id', 'review_score']],
    on='order_id',
    how='left'  # Pedidos sem review permanecem na base; a média ignora NaN
)

# 3) Agrupa por vendedor e calcula os principais indicadores
# - total_pedidos: quantidade de pedidos únicos
# - media_nota: média das notas recebidas
# - receita_total: soma dos preços vendidos
seller_stats = (
    items_with_review
    .groupby('seller_id')
    .agg(
        total_pedidos=('order_id', 'nunique'),
        media_nota=('review_score', 'mean'),
        receita_total=('price', 'sum')
    )
    .reset_index()
)

# Conferência: quantos vendedores ficaram na base final
print(f'Total de vendedores com dados: {len(seller_stats):,}')

# Mostra as primeiras linhas para validar estrutura e colunas
seller_stats.head()

Total de vendedores com dados: 2,970


,seller_id,total_pedidos,media_nota,receita_total
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3.666667,2685.00
1,001cca7ae9ae17fb1caed9dfb1094831,195,3.965368,24487.03
2,002100f778ceb8431b7a1020ff7ab48f,50,4.037037,1216.60
3,003554e2dce176b5555353e4f3555ac8,1,5.000000,120.00
4,004c9cd9d87a3c30c522c48c4fc07416,156,4.132530,19569.73


---
## 4. Segmentação de detratores e diagnóstico inicial

Nesta etapa, classificamos os vendedores por nota média e comparamos o grupo de detratores com os demais.

**Regra adotada nesta versão:**
- `detrator = True` quando `media_nota <= 3`.

In [68]:
# ------------------- Segmentação de vendedores -------------------
# Regra usada: vendedor com média de nota <= 3 é classificado como detrator.

# 1) Cria coluna booleana (True/False) para identificar detratores
seller_stats['detrator'] = seller_stats['media_nota'] <= 3

# 2) Separa em dois grupos para comparação
# .copy() evita alertas de modificação indireta em pandas
detratores = seller_stats[seller_stats['detrator']].copy()
nao_detratores = seller_stats[~seller_stats['detrator']].copy()

# 3) Mostra quantos vendedores existem em cada grupo
print('Distribuição de vendedores por grupo:')
print(seller_stats['detrator'].value_counts())

# 4) Resumo estatístico do grupo detrator
# Importante para leigos: na linha "count", o valor indica quantos vendedores
# entraram nesse grupo para cada métrica analisada.
print('\nResumo descritivo - detratores:')
display(detratores[['total_pedidos', 'media_nota', 'receita_total']].describe())

# 5) Resumo estatístico do grupo não detrator
print('\nResumo descritivo - não detratores:')
display(nao_detratores[['total_pedidos', 'media_nota', 'receita_total']].describe())

Distribuição de vendedores por grupo:
detrator
False    2699
True      271
Name: count, dtype: int64

Resumo descritivo - detratores:


,total_pedidos,media_nota,receita_total
count,271.000000,271.000000,271.000000
mean,5.184502,2.234119,1154.328967
std,16.077218,0.800038,3695.091642
min,1.000000,1.000000,6.900000
25%,1.000000,1.400000,85.425000
50%,2.000000,2.500000,199.000000
75%,4.000000,3.000000,753.380000
max,187.000000,3.000000,38990.720000



Resumo descritivo - não detratores:


,total_pedidos,media_nota,receita_total
count,2699.000000,2694.000000,2699.000000
mean,35.719155,4.342534,4782.299289
std,110.070338,0.483247,14521.367738
min,1.000000,3.034483,6.500000
25%,3.000000,4.000000,266.000000
50%,8.000000,4.333333,987.700000
75%,26.000000,4.750000,3792.500000
max,1819.000000,5.000000,226987.930000


In [63]:
# ------------------- Resumo executivo do impacto -------------------
# Aqui transformamos a segmentação em uma tabela de impacto para narrativa executiva.

# 1) Soma métricas por grupo (detrator x não detrator)
final_detratores = (
    seller_stats
    .groupby('detrator', as_index=False)
    .agg(
        total_pedidos=('total_pedidos', 'sum'),
        receita_total=('receita_total', 'sum'),
        vendedores=('seller_id', 'count')
    )
)

# 2) Calcula participação percentual em receita
# Fórmula: (valor do grupo / valor total) * 100
total_receita = final_detratores['receita_total'].sum()
final_detratores['%Receita_total'] = (final_detratores['receita_total'] / total_receita) * 100

# 3) Calcula participação percentual em pedidos
total_pedido = final_detratores['total_pedidos'].sum()
final_detratores['%Pedido_total'] = (final_detratores['total_pedidos'] / total_pedido) * 100

# 4) Calcula participação percentual na quantidade de vendedores
total_vendedores = final_detratores['vendedores'].sum()
final_detratores['%Vendedor_total'] = (final_detratores['vendedores'] / total_vendedores) * 100

# 5) Arredonda resultados para facilitar leitura em apresentação
final_detratores = final_detratores.round({
    'receita_total': 2,
    '%Receita_total': 2,
    '%Pedido_total': 2,
    '%Vendedor_total': 2,
})

# 6) Exibe a tabela final consolidada
final_detratores

,detrator,total_pedidos,receita_total,vendedores,%Receita_total,%Pedido_total,%Vendedor_total
0,False,96406,12907425.78,2699,97.63,98.56,90.88
1,True,1405,312823.15,271,2.37,1.44,9.12


---
## 5. Leitura executiva dos resultados

A tabela `final_detratores` permite responder, de forma objetiva, três perguntas:

1. Qual a participação dos detratores no volume de pedidos (`%Pedido_total`)?
2. Qual o impacto financeiro desse grupo (`%Receita_total`)?
3. Qual o tamanho do grupo dentro da base de vendedores (`%Vendedor_total`)?

Com essa estrutura, a narrativa final e os gráficos podem ser construídos diretamente a partir de um único resumo consolidado.